# AI-Driven Cloud Cost Optimization — Proof of Concept

**Goal:** show that an LLM, given a resource description and a small set of grounding facts, can suggest a reasonable and cheaper alternative — including cross-architecture suggestions (e.g. EC2 -> Lambda), not just resizing within the same service.

**Important note on data used in this notebook:**
All pricing/service figures below (`MOCK_pricing_knowledge`) are **illustrative placeholders**, not verified against live AWS pricing. They were drafted to be *plausible* so we could test whether the reasoning pipeline works — they are **not** meant to be trusted as real costs.

In the full project, this placeholder dictionary will be replaced by:
- Real, current data pulled from the **AWS Pricing API**
- A **retrieval step (RAG)** that automatically finds relevant facts for a given resource, instead of a human manually sorting facts into categories
- A **validation engine** that numerically re-checks any LLM cost claim against the real pricing data before it's trusted

This PoC exists only to prove the reasoning step itself is sound — not to prove the numbers are accurate yet.

In [1]:
!pip install python-hcl2 --break-system-packages -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.0/99.0 kB 4.1 MB/s eta 0:00:00


In [2]:
sample_tf_content = '''
provider "aws" {
  region = "us-east-1"
}

resource "aws_instance" "web" {
  ami           = "ami-0123456789"
  instance_type = "t3.large"

  root_block_device {
    volume_size = 100
  }
}
'''
with open("sample.tf", "w") as f:
    f.write(sample_tf_content)

print("Wrote sample.tf")

Wrote sample.tf


In [3]:
from tf_json_convert import terraform_to_resources

parsed_resources = terraform_to_resources("sample.tf")
print(f"Parsed {len(parsed_resources)} resource(s):")
for r in parsed_resources:
    print(r)

Parsed 1 resource(s):
{'instance_type': 't3.large', 'storage_gb': 100, 'ami': 'ami-0123456789', 'availability_zone': None, 'key_name': None, 'subnet_id': None, 'security_group_ids': [], 'volume_type': None, 'function_name': None, 'runtime': None, 'handler': None, 'memory_mb': None, 'timeout_seconds': None, 'role': None, 'filename': None, 'environment': {}, 'tags': {}, 'resource_type': 'aws_instance', 'resource_name': 'web', 'region': 'us-east-1'}


### Bridge step:
attach a placeholder cost We attach a mock monthly cost here so `build_prompt()` has what it needs. In the full project this comes from real AWS billing data instead of this lookup table.

In [4]:
def attach_mock_cost(resource: dict) -> dict:
    MOCK_INSTANCE_COSTS = {
        "t3.medium": 30,
        "t3.large": 60,
        "t3.xlarge": 120,
    }
    resource = dict(resource)  # don't mutate the original
    resource["monthly_cost_usd"] = MOCK_INSTANCE_COSTS.get(
        resource.get("instance_type"), 50  # fallback placeholder if type unknown
    )
    return resource


resources_with_cost = [attach_mock_cost(r) for r in parsed_resources]
print("With placeholder costs attached:")
for r in resources_with_cost:
    print(r)

With placeholder costs attached:
{'instance_type': 't3.large', 'storage_gb': 100, 'ami': 'ami-0123456789', 'availability_zone': None, 'key_name': None, 'subnet_id': None, 'security_group_ids': [], 'volume_type': None, 'function_name': None, 'runtime': None, 'handler': None, 'memory_mb': None, 'timeout_seconds': None, 'role': None, 'filename': None, 'environment': {}, 'tags': {}, 'resource_type': 'aws_instance', 'resource_name': 'web', 'region': 'us-east-1', 'monthly_cost_usd': 60}


## 2. Placeholder pricing/grounding facts

**MOCK DATA — not verified against AWS's real, current pricing.** Used only to test whether the LLM can reason well when given *some* grounding context.

In [5]:
# MOCK / PLACEHOLDER pricing knowledge — for PoC reasoning tests only.
# Real project version: pull this data live from the AWS Pricing API instead.
MOCK_pricing_knowledge = {
    "aws_instance": """
[PLACEHOLDER FIGURES]
- t3.medium: ~$30/month on-demand
- t3.large: ~$60/month on-demand
- 1-year Savings Plan: ~20% discount on compute
- Reserved Instances: up to 40% discount for 1-year commit
- AWS Lambda: serverless compute, pay per invocation + duration, good for event-driven/intermittent workloads. ~$0.20 per million requests + ~$0.0000166667 per GB-second.
- AWS Fargate: serverless compute for containers, pay for vCPU/memory consumed. ~$0.04048 per vCPU-hour + ~$0.004445 per GB-hour.
""",
    "aws_s3_bucket": """
[PLACEHOLDER FIGURES]
- S3 Standard: ~$0.023/GB/month
- S3 Intelligent-Tiering: same cost, auto-moves cold data to cheaper tiers
- S3 Glacier: ~$0.004/GB/month for rarely-accessed data
""",
    "aws_db_instance": """
[PLACEHOLDER FIGURES]
- RDS on-demand: varies by instance class, ~$0.10-0.50/hr
- RDS Reserved Instances: up to 40% discount for 1-year commit
- Aurora Serverless: pay only for actual usage, good for spiky workloads
- DynamoDB (on-demand): pay per request, good for key-value access patterns instead of a full relational DB
"""
}

print("Loaded MOCK_pricing_knowledge (placeholder figures only, not live data).")

Loaded MOCK_pricing_knowledge (placeholder figures only, not live data).


## 3. Prompt builder

In [6]:
def build_prompt(resource):
    resource_type = resource["resource_type"]
    facts = MOCK_pricing_knowledge.get(
        resource_type,
        "No specific pricing facts available — use general AWS knowledge."
    )

    prompt = f"""You are a cloud cost optimization assistant.

Given this AWS resource and its current monthly cost, suggest ONE cheaper
alternative that achieves the same function. Consider architectural changes
like serverless or containers if the workload characteristics align, not just
resizing within the same resource type.

Resource: {resource}

Relevant pricing facts (NOTE: these are placeholder figures for testing purposes):
{facts}

Respond ONLY with valid JSON in this exact format, no other text:
{{"suggested_alternative": "...", "reason": "...", "estimated_monthly_cost_usd": 0}}
"""
    return prompt

In [ ]:
import re
import json
from tf_json_convert import resources_to_terraform


def apply_suggestion_to_resource(original_resource: dict, suggestion: dict):
    """
    Applies an LLM suggestion back onto the original parsed resource dict,
    producing a new resource dict ready for resources_to_terraform().

    Handles two cases now that Omar Tarek's script supports Lambda:
      1. Suggestion is still an EC2 instance, just a different size
         (e.g. "t3.medium") -> update instance_type in place.
      2. Suggestion is AWS Lambda -> convert to an aws_lambda_function
         dict, filling in reasonable defaults for required fields the
         LLM doesn't provide (runtime, handler, role, etc.). These
         defaults are placeholders a real user MUST review/replace
         before `terraform apply` — see role/filename below.

    Returns None if the suggestion doesn't match a supported conversion
    (e.g. Fargate — not yet supported by Omar Tarek's writer).
    """
    alt_text = suggestion.get("suggested_alternative", "").lower()

    # Case 1: still an EC2 instance, just resized
    ec2_match = re.search(r"t3\.\w+", alt_text)
    if ec2_match and "lambda" not in alt_text and "fargate" not in alt_text:
        new_resource = dict(original_resource)
        new_resource["instance_type"] = ec2_match.group(0)
        new_resource["monthly_cost_usd"] = suggestion.get(
            "estimated_monthly_cost_usd", new_resource.get("monthly_cost_usd")
        )
        return new_resource

    # Case 2: suggestion is AWS Lambda -> build a valid Lambda resource dict
    if "lambda" in alt_text:
        new_resource = dict(original_resource)
        new_resource["resource_type"] = "aws_lambda_function"
        new_resource["function_name"] = f"{original_resource.get('resource_name', 'app')}-fn"
        new_resource["runtime"] = "python3.12"          # placeholder — confirm real runtime needed
        new_resource["handler"] = "index.handler"        # placeholder — depends on actual code
        new_resource["memory_mb"] = 512                  # placeholder — tune based on real workload
        new_resource["timeout_seconds"] = 60              # placeholder
        new_resource["role"] = "arn:aws:iam::123456789012:role/lambda-exec"  # PLACEHOLDER — must be a real IAM role ARN
        new_resource["filename"] = None                  # writer emits a REPLACE_ME.zip TODO for this
        new_resource["instance_type"] = None
        new_resource["monthly_cost_usd"] = suggestion.get("estimated_monthly_cost_usd")
        return new_resource

    # Case 3: Fargate or anything else not yet supported by the writer
    return None

## 5b. Set up the LLM API client

Run this once per session (must be re-run after any Colab runtime restart, since restarting wipes all variables).

In [ ]:
from google.colab import userdata
from openai import OpenAI

try:
    groq_key = userdata.get('GROQ_API_KEY')
except Exception:
    raise ValueError("Please add your 'GROQ_API_KEY' to the Colab Secrets (key icon) tab first.")

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_key
)
print("Groq client ready.")

In [12]:
# Run the round-trip on the resource from section 6
resource = resources_with_cost[0]
prompt = build_prompt(resource)
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}]
)

raw_text = response.choices[0].message.content
clean_text = raw_text.replace("```json", "").replace("```", "").strip()
suggestion = json.loads(clean_text)
print("LLM suggestion:", suggestion)

updated_resource = apply_suggestion_to_resource(resource, suggestion)

if updated_resource is None:
    print("\nSuggestion isn't an EC2 instance-type change — the current Terraform")
    print("writer can't express this yet (e.g. it suggested Lambda/Fargate).")
else:
    print("\nUpdated resource dict:")
    print(updated_resource)

    new_tf_code = resources_to_terraform([updated_resource])
    print("\n--- Generated Terraform code ---")
    print(new_tf_code)

NameError: name 'client' is not defined

In [ ]:
test_case_tf_content = '''
provider "aws" {
  region = "us-east-1"
}

resource "aws_instance" "batch_worker" {
  ami           = "ami-0abcdef1234567890"
  instance_type = "t3.xlarge"

  root_block_device {
    volume_size = 50
  }

  tags = {
    Name        = "nightly-batch-worker"
    Environment = "production"
  }
}
'''

with open("test_case_oversized.tf", "w") as f:
    f.write(test_case_tf_content)

print("Wrote test_case_oversized.tf")

Wrote test_case_oversized.tf


In [ ]:
# Parse -> attach mock cost -> reason -> apply suggestion -> generate new Terraform

parsed = terraform_to_resources("test_case_oversized.tf")
resource = attach_mock_cost(parsed[0])
print("Parsed + costed resource:")
print(resource)

prompt = build_prompt(resource) + "\n\nNote: for this resource, prefer suggesting a smaller EC2 instance type if a reasonable one exists, before suggesting a full architecture change."
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}]
)

raw_text = response.choices[0].message.content
clean_text = raw_text.replace("```json", "").replace("```", "").strip()
suggestion = json.loads(clean_text)
print("\nLLM suggestion:", suggestion)

updated_resource = apply_suggestion_to_resource(resource, suggestion)

if updated_resource is None:
    print("\nSuggestion isn't an EC2 instance-type change — the current Terraform")
    print("writer can't express this yet (e.g. it suggested Lambda/Fargate).")
    print("Re-run this cell — the LLM's answer can vary between runs.")
else:
    print("\nUpdated resource dict:")
    print(updated_resource)

    new_tf_code = resources_to_terraform([updated_resource])
    print("\n--- Generated Terraform code ---")
    print(new_tf_code)

Parsed + costed resource:
{'resource_type': 'aws_instance', 'resource_name': 'batch_worker', 'instance_type': 't3.xlarge', 'region': 'us-east-1', 'storage_gb': 50, 'ami': 'ami-0abcdef1234567890', 'availability_zone': None, 'key_name': None, 'subnet_id': None, 'security_group_ids': [], 'volume_type': None, 'tags': {'Name': 'nightly-batch-worker', 'Environment': 'production'}, 'monthly_cost_usd': 120}

LLM suggestion: {'suggested_alternative': 't3.large', 'reason': 'Downsizing from t3.xlarge to t3.large reduces costs while still providing sufficient resources for the batch worker', 'estimated_monthly_cost_usd': 60}

Updated resource dict:
{'resource_type': 'aws_instance', 'resource_name': 'batch_worker', 'instance_type': 't3.large', 'region': 'us-east-1', 'storage_gb': 50, 'ami': 'ami-0abcdef1234567890', 'availability_zone': None, 'key_name': None, 'subnet_id': None, 'security_group_ids': [], 'volume_type': None, 'tags': {'Name': 'nightly-batch-worker', 'Environment': 'production'}, 'mon

## 7. Summary / what this PoC proves (and doesn't)

**Proves:**
- The LLM can take a structured resource description + a small set of grounding facts and produce a sensible, structured (JSON) suggestion.
- When given workload context, it can reason about cross-architecture alternatives (e.g. EC2 -> Lambda), not just same-service resizing.
- The pipeline works end-to-end from a real `.tf` file (via Omar Tarek's parser) through to a suggestion, not just from hand-typed dicts.

**Does NOT prove (by design — out of scope for PoC):**
- That the pricing figures are accurate (`MOCK_pricing_knowledge` and `MOCK_INSTANCE_COSTS` are both placeholder data — only the Terraform *parsing* is real, not the billing data).
- That suggestions are automatically retrieved (facts here are manually pre-sorted per resource type, not fetched via RAG).
- That any suggestion is validated numerically before being trusted (no validation engine yet).

**Planned for the full project:**
- Replace `MOCK_pricing_knowledge` with real data pulled from the **AWS Pricing API**.
- Add a retrieval step (RAG) so relevant facts are found automatically for any resource, not hand-sorted by us.
- Add a validation engine that re-checks the LLM's cost claim against real pricing data before any suggestion is accepted or applied.